# Lab 2 — Baseline com múltiplos datasets

Neste notebook vamos rodar o mesmo pipeline de baseline para 4 datasets diferentes.

A ideia é simples: para cada dataset na lista `keys_to_run`, o código vai:
1. baixar e carregar os dados
2. limpar valores ausentes e aplicar `dropna` antes do split
3. treinar dois modelos baseline (um linear e um de floresta)
4. imprimir as métricas e salvar os resultados

No final, uma tabela compara os melhores modelos de cada dataset.

In [3]:
# 1) Bibliotecas
from __future__ import annotations

import json
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [4]:
# 2) Metadados dos datasets e funcoes  que nos ajudam a baixar, extrair e normalizar os dados

DATASETS = {
    "adult": {"task": "classification", "target": "income"},
    "bank_marketing": {"task": "classification", "target": "y"},
    "air_quality_uci": {"task": "regression", "target": "C6H6(GT)"},
    "communities_crime": {"task": "regression", "target": "ViolentCrimesPerPop"},
}

# Tokens que representam valores ausentes
NUMERIC_MISSING_VALUES = [-200, -200.0]

STRING_MISSING_VALUES = [
    "?", " ?", "? ", "NA", "N/A", "na", "n/a", "NaN", "nan", "", " ",
    "unknown", "Unknown", "-200",
]


def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def download_file(url: str, output_path: Path) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if not output_path.exists():
        print(f"[download] {url}")
        urlretrieve(url, output_path)
    else:
        print(f"[cache] {output_path}")
    return output_path


def unzip_file(zip_path: Path, extract_dir: Path) -> Path:
    extract_dir.mkdir(parents=True, exist_ok=True)
    marker = extract_dir / ".extracted"
    if not marker.exists():
        print(f"[extract] {zip_path}")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_dir)
        marker.write_text("ok\n", encoding="utf-8")
    else:
        print(f"[cache] {extract_dir}")
    return extract_dir


def normalize_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Substitui tokens de missing por NaN, coluna a coluna, respeitando o dtype.
    
    - Colunas numericas: substitui -200 e -200.0
    - Colunas de texto: faz strip e substitui tokens como '?', 'unknown', etc.
    Essa abordagem evita erros do pandas 2.x ao misturar int e str no replace().
    -- TODO: verificar se tem outros simbolos estranhos nos dataset da atividade (pedir par aos alunos checarem isso)
    """
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].replace(NUMERIC_MISSING_VALUES, np.nan)
        else:
            df[col] = df[col].astype("string").str.strip()
            df[col] = df[col].replace(STRING_MISSING_VALUES, pd.NA)
    return df

In [5]:
# 3) Funcoes. para baixar os datasets automaticamente

# fiz alguns pequenos ajustes caso a caso para facilitar a leitura e evitar erros de parsing

def load_adult(data_dir: Path) -> tuple[pd.DataFrame, str, str]:
    base = data_dir / "adult"
    data_file = download_file(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
        base / "adult.data",
    )
    test_file = download_file(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test",
        base / "adult.test",
    )

    columns = [
        "age", "workclass", "fnlwgt", "education", "education_num", "marital_status",
        "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
        "hours_per_week", "native_country", "income",
    ]

    train_df = pd.read_csv(data_file, header=None, names=columns, skipinitialspace=True, na_values=["?", " ?"])
    test_df = pd.read_csv(test_file, header=None, names=columns, skiprows=1, skipinitialspace=True, na_values=["?", " ?"])

    df = pd.concat([train_df, test_df], ignore_index=True)
    df["income"] = df["income"].astype(str).str.replace(".", "", regex=False).str.strip()
    return df, "income", "classification"


def load_bank_marketing(data_dir: Path) -> tuple[pd.DataFrame, str, str]:
    base = data_dir / "bank_marketing"
    zip_path = download_file(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank.zip",
        base / "bank.zip",
    )
    extract_dir = unzip_file(zip_path, base / "extracted")
    df = pd.read_csv(extract_dir / "bank-full.csv", sep=";")
    return df, "y", "classification"


def load_air_quality_uci(data_dir: Path) -> tuple[pd.DataFrame, str, str]:
    base = data_dir / "air_quality_uci"
    zip_path = download_file(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/00360/AirQualityUCI.zip",
        base / "AirQualityUCI.zip",
    )
    extract_dir = unzip_file(zip_path, base / "extracted")
    candidates = list(extract_dir.rglob("AirQualityUCI.csv"))
    if not candidates:
        raise FileNotFoundError("Nao encontrei AirQualityUCI.csv dentro do zip.")

    df = pd.read_csv(candidates[0], sep=";", decimal=",")
    df = df.dropna(axis=1, how="all").dropna(axis=0, how="all")
    for col in ["Date", "Time"]:
        if col in df.columns:
            df = df.drop(columns=[col])
    return df, "C6H6(GT)", "regression"


def load_communities_crime(data_dir: Path) -> tuple[pd.DataFrame, str, str]:
    base = data_dir / "communities_crime"
    data_file = download_file(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/communities/communities.data",
        base / "communities.data",
    )
    df = pd.read_csv(data_file, header=None, na_values=["?"])

    cols = [f"col_{i}" for i in range(df.shape[1])]
    cols[-1] = "ViolentCrimesPerPop"
    df.columns = cols
    # colunas  com muito problema
    df = df.drop(columns=[c for c in ["col_0", "col_1", "col_2", "col_3", "col_4"] if c in df.columns])
    return df, "ViolentCrimesPerPop", "regression"


def load_dataset(key: str, data_dir: Path) -> tuple[pd.DataFrame, str, str]:
    if key == "adult":
        return load_adult(data_dir)
    if key == "bank_marketing":
        return load_bank_marketing(data_dir)
    if key == "air_quality_uci":
        return load_air_quality_uci(data_dir)
    if key == "communities_crime":
        return load_communities_crime(data_dir)
    raise ValueError(f"Key desconhecida: {key}")

In [6]:
# 4) Funcoes para separar  treino e teste

def split_X_y(df: pd.DataFrame, target: str) -> tuple[pd.DataFrame, pd.Series]:
    if target not in df.columns:
        raise ValueError(f"Target '{target}' nao esta no dataset.")
    return df.drop(columns=[target]), df[target]


def build_preprocessor(X: pd.DataFrame, scale_numeric: bool) -> ColumnTransformer:
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler() if scale_numeric else "passthrough", numeric_cols),
            ("cat", make_one_hot_encoder(), categorical_cols),
        ],
        remainder="drop",
    )


def run_classification(X: pd.DataFrame, y: pd.Series, test_size: float, random_state: int) -> dict:
    class_counts = y.value_counts(dropna=False)
    stratify = y if class_counts.min() >= 2 else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=stratify
    )

    models = {
        "logistic_regression": Pipeline([
            ("preprocess", build_preprocessor(X_train, scale_numeric=True)),
            ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
        ]),
        "random_forest": Pipeline([
            ("preprocess", build_preprocessor(X_train, scale_numeric=False)),
            ("model", RandomForestClassifier(
                n_estimators=200,
                random_state=random_state,
                n_jobs=-1,
                class_weight="balanced_subsample",
            )),
        ]),
    }

    results = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        metrics = {
            "accuracy": float(accuracy_score(y_test, pred)),
            "macro_f1": float(f1_score(y_test, pred, average="macro")),
            "weighted_f1": float(f1_score(y_test, pred, average="weighted")),
        }
        results[name] = metrics

        print(f"\n[model] {name}")
        print(metrics)
        print("Matriz de confusao:")
        print(confusion_matrix(y_test, pred))
        print(classification_report(y_test, pred))

    return results


def run_regression(X: pd.DataFrame, y: pd.Series, test_size: float, random_state: int) -> dict:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    models = {
        "ridge_regression": Pipeline([
            ("preprocess", build_preprocessor(X_train, scale_numeric=True)),
            ("model", Ridge(alpha=1.0)),
        ]),
        "random_forest": Pipeline([
            ("preprocess", build_preprocessor(X_train, scale_numeric=False)),
            ("model", RandomForestRegressor(n_estimators=200, random_state=random_state, n_jobs=-1)),
        ]),
    }

    results = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        mse = mean_squared_error(y_test, pred)
        metrics = {
            "mae": float(mean_absolute_error(y_test, pred)),
            "rmse": float(np.sqrt(mse)),
            "r2": float(r2_score(y_test, pred)),
        }
        results[name] = metrics
        print(f"\n[model] {name}")
        print(metrics)

    return results


def run_one_dataset(key: str, out_dir: str = "data", test_size: float = 0.2, random_state: int = 42) -> dict:
    data_dir = Path(out_dir)
    df, target, task = load_dataset(key, data_dir=data_dir)

    print("\n" + "=" * 80)
    print(f"Dataset: {key} | Task: {task} | Target: {target}")
    print("=" * 80)
    print("Shape original:", df.shape)

    df = normalize_missing_values(df)
    rows_before = len(df)
    df_clean = df.dropna(axis=0, how="any").copy()
    rows_after = len(df_clean)
    print("Linhas removidas no dropna:", rows_before - rows_after)

    if rows_after == 0:
        raise RuntimeError("Dataset vazio apos dropna.")

    X, y = split_X_y(df_clean, target)

    if task == "classification":
        print("Distribuicao da target:")
        print(y.value_counts())
        results = run_classification(X, y, test_size=test_size, random_state=random_state)
    else:
        y = pd.to_numeric(y, errors="coerce")
        valid = y.notna()
        X = X.loc[valid].copy()
        y = y.loc[valid].copy()
        print("Resumo da target:\n", y.describe())
        results = run_regression(X, y, test_size=test_size, random_state=random_state)

    out_path = data_dir / f"{key}_baseline_results.json"
    out_path.write_text(
        json.dumps(
            {
                "key": key,
                "task": task,
                "target": target,
                "shape_original": list(df.shape),
                "shape_after_dropna": list(df_clean.shape),
                "test_size": test_size,
                "random_state": random_state,
                "results": results,
            },
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )
    print("Resultados salvos em:", out_path)

    return {
        "key": key,
        "task": task,
        "rows_original": int(df.shape[0]),
        "rows_after_dropna": int(df_clean.shape[0]),
        "results": results,
    }

In [7]:
# 5) Configuracao da execucao 

# Pessoal, caso fique pesado, teste em um dataet primeiro, faca sample dos dados, ou diminua o n_estimators do random forest para 50 ou 100, etc

datasets_experiment = [
    "adult",
    "bank_marketing",
    "air_quality_uci",
    "communities_crime",
]

# Parametros globais
out_dir = "data"
test_size = 0.2
random_state = 42

datasets_experiment

['adult', 'bank_marketing', 'air_quality_uci', 'communities_crime']

In [8]:
# 6) Loop principal

all_runs = []

for key in datasets_experiment:
    try:
        run_info = run_one_dataset(
            key=key,
            out_dir=out_dir,
            test_size=test_size,
            random_state=random_state,
        )
        all_runs.append(run_info)
    except Exception as e:
        print(f"\n[ERRO] key={key}: {e}")
        all_runs.append({"key": key, "task": DATASETS[key]["task"], "error": str(e)})


# --- Monta tabelas separadas por tipo de tarefa ---

classification_rows = []
regression_rows = []

for r in all_runs:
    base = {"key": r["key"], "best_model": "", "error": r.get("error", "")}

    if "error" in r and "results" not in r:
        # Dataset falhou: adiciona na tabela correta com metricas vazias
        if r.get("task") == "classification":
            classification_rows.append({**base, "accuracy": None, "macro_f1": None, "weighted_f1": None})
        else:
            regression_rows.append({**base, "mae": None, "rmse": None, "r2": None})
        continue

    if r["task"] == "classification":
        best = max(r["results"], key=lambda m: r["results"][m]["weighted_f1"])
        m = r["results"][best]
        classification_rows.append({
            "key": r["key"],
            "best_model": best,
            "accuracy": round(m["accuracy"], 4),
            "macro_f1": round(m["macro_f1"], 4),
            "weighted_f1": round(m["weighted_f1"], 4),
            "error": "",
        })
    else:
        best = max(r["results"], key=lambda m: r["results"][m]["r2"])
        m = r["results"][best]
        regression_rows.append({
            "key": r["key"],
            "best_model": best,
            "mae": round(m["mae"], 4),
            "rmse": round(m["rmse"], 4),
            "r2": round(m["r2"], 4),
            "error": "",
        })

# Exibe  resultados e salva CSVs 

data_path = Path(out_dir)

if classification_rows:
    clf_df = pd.DataFrame(classification_rows, columns=["key", "best_model", "accuracy", "macro_f1", "weighted_f1", "error"])
    print("\n=== Classificacao ===")
    display(clf_df)
    clf_path = data_path / "classification_summary.csv"
    clf_df.to_csv(clf_path, index=False)
    print("Salvo em:", clf_path)

if regression_rows:
    reg_df = pd.DataFrame(regression_rows, columns=["key", "best_model", "mae", "rmse", "r2", "error"])
    print("\n=== Regressao ===")
    display(reg_df)
    reg_path = data_path / "regression_summary.csv"
    reg_df.to_csv(reg_path, index=False)
    print("Salvo em:", reg_path)

[download] https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data
[download] https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test

Dataset: adult | Task: classification | Target: income
Shape original: (48842, 15)
Linhas removidas no dropna: 3620
Distribuicao da target:
income
<=50K    34014
>50K     11208
Name: count, dtype: Int64

[model] logistic_regression
{'accuracy': 0.8091763405196241, 'macro_f1': 0.7742666199497235, 'weighted_f1': 0.8190299698286431}
Matriz de confusao:
[[5438 1365]
 [ 361 1881]]
              precision    recall  f1-score   support

       <=50K       0.94      0.80      0.86      6803
        >50K       0.58      0.84      0.69      2242

    accuracy                           0.81      9045
   macro avg       0.76      0.82      0.77      9045
weighted avg       0.85      0.81      0.82      9045


[model] random_forest
{'accuracy': 0.8527363184079602, 'macro_f1': 0.7892272326489482, 'weighted_f1': 0.8475686103220306

,key,best_model,accuracy,macro_f1,weighted_f1,error
0,adult,random_forest,0.8527,0.7892,0.8476,
1,bank_marketing,random_forest,0.8464,0.7575,0.8375,


Salvo em: data\classification_summary.csv

=== Regressao ===


,key,best_model,mae,rmse,r2,error
0,air_quality_uci,random_forest,0.0694,0.2302,0.9991,
1,communities_crime,random_forest,0.1110,0.1458,0.7041,


Salvo em: data\regression_summary.csv


# Criação de tabela resumo

In [10]:
df_tabela_resumo = pd.DataFrame(all_runs)
df_tabela_resumo.to_csv(data_path / "tabela_resumo_completa.csv", index=False)